# 🐼 Pandas: полный практический туториал

**Pandas** — главная библиотека Python для анализа табличных данных.  
Этот ноутбук рассчитан на новичков: от первых шагов до реальных сценариев очистки и агрегации данных.

**Чему научитесь:**
- Создавать данные (Series, DataFrame) и читать файлы (CSV, Excel)
- Исследовать датасет (head, info, describe, sample)
- Выбирать, фильтровать и сортировать данные
- Работать с пропусками, дубликатами и текстом
- Преобразовывать типы, включая даты и категории
- Группировать, агрегировать и строить сводные таблицы
- Объединять таблицы и менять их форму
- Визуализировать данные и сохранять результаты

**Установка (если нужно):**
```bash
pip install pandas openpyxl matplotlib
```

## 1. Импорт и базовая структура: Series

`Series` — это одномерный массив с подписями (индексами). Представьте колонку Excel с именованными ячейками.

In [ ]:
import pandas as pd
import numpy as np

print("Версия pandas:", pd.__version__)

# Series из списка
s = pd.Series([10, 20, 30, 40], index=['a', 'b', 'c', 'd'])
print(s)
print("
Сумма:", s.sum(), "| Среднее:", s.mean(), "| Максимум:", s.max())

# Математические операции векторизованы — работают над всей колонкой сразу
print("
Каждый элемент * 2:")
print(s * 2)

## 2. DataFrame — таблица из строк и столбцов

Создадим вручную из словаря. На практике чаще загружают из файла.

In [ ]:
df = pd.DataFrame({
    'имя': ['Анна', 'Борис', 'Вера', 'Григорий', 'Дарья'],
    'возраст': [25, 32, 28, 45, 31],
    'город': ['Москва', 'СПб', 'Казань', 'Москва', 'СПб'],
    'зарплата': [90000, 120000, 85000, 150000, 110000],
    'работает': [True, True, False, True, True]
})
df

In [ ]:
# Метаданные таблицы
print("Размер:", df.shape)          # (строки, столбцы)
print("Колонки:", df.columns.tolist())
print("Типы данных:")
print(df.dtypes)

## 3. Загрузка данных из файлов

На практике данные берутся из CSV, Excel, баз данных или API.  
Ниже — загрузка классического датасета «Титаник». Если интернета нет, создадим похожий синтетический набор.

In [ ]:
try:
    url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
    titanic = pd.read_csv(url)
    print("✅ Загружено из интернета")
except Exception as e:
    # Запасной вариант без интернета
    rng = np.random.default_rng(42)
    n = 600
    titanic = pd.DataFrame({
        'PassengerId': range(1, n + 1),
        'Survived': rng.integers(0, 2, n),
        'Pclass': rng.integers(1, 4, n),
        'Name': [f'Passenger {i}' for i in range(n)],
        'Sex': rng.choice(['male', 'female'], n),
        'Age': np.round(rng.normal(30, 14, n), 1),
        'SibSp': rng.integers(0, 6, n),
        'Parch': rng.integers(0, 3, n),
        'Fare': np.round(np.abs(rng.normal(50, 40, n)), 2),
        'Embarked': rng.choice(['S', 'C', 'Q'], n),
        'Cabin': np.nan,
    })
    print("⚠️ Сгенерирован локальный датасет (интернета не было)")

print(f"Строк: {titanic.shape[0]}, столбцов: {titanic.shape[1]}")

In [ ]:
# Чтение Excel (требуется openpyxl):
# df_excel = pd.read_excel('data.xlsx', sheet_name='Лист1')

# Основные параметры read_csv:
# pd.read_csv('file.csv', sep=';', encoding='utf-8', decimal=',', nrows=1000)


## 4. Первый взгляд на данные

Всегда начинайте с этих команд — они покажут структуру, проблемы и аномалии.

In [ ]:
# Первые и последние строки
display(titanic.head(3))
display(titanic.tail(2))

In [ ]:
# Случайная выборка — чтобы не смотреть только на «верхушку»
titanic.sample(5, random_state=1)

In [ ]:
# Типы данных и пропуски
titanic.info()

In [ ]:
# Числовая статистика
titanic.describe().round(2)

In [ ]:
# Уникальные значения по каждому столбцу (быстрый обзор)
titanic.nunique()

## 5. Индексация: как выбрать данные

Pandas предлагает несколько способов выбора. Главное правило: используйте **один явный метод**, не смешивая их в одной цепочке.

In [ ]:
# [] — быстрый выбор: ОДИН столбец (Series) или список столбцов (DataFrame)
print(type(titanic['Name']))           # Series
print(type(titanic[['Name', 'Age']]))  # DataFrame

# Но НЕ используйте [] для сложных условий — путаница гарантирована

In [ ]:
# .loc[] — выбор по МЕТКАМ (имена столбцов, метки индекса)
# Включает обе границы среза [0:2] означает 0, 1, 2

titanic.loc[0:2, ['Name', 'Sex', 'Age']]

In [ ]:
# .iloc[] — выбор по ПОЗИЦИЯМ (как в numpy)
# НЕ включает правую границу: [0:3] означает 0, 1, 2

titanic.iloc[0:3, 0:5]

In [ ]:
# .at[] и .iat[] — для извлечения ОДНОГО значения (быстрее)
print("Имя первого пассажира:", titanic.at[0, 'Name'])
print("Возраст третьего:", titanic.iat[2, 5])  # позиция столбца Age (зависит от порядка)

In [ ]:
# Сброс и установка индекса
# Иногда удобно сделать PassengerId индексом, чтобы не путать с порядковым номером строки

titanic_indexed = titanic.set_index('PassengerId')
print(titanic_indexed.head(2))

# Вернуть индекс в столбец
titanic_indexed.reset_index().head(2)

## 6. Фильтрация (условная индексация)

Условие создаёт **маску** — Series из `True/False`. Передав её в `[]`, оставляем только нужные строки.

In [ ]:
# Одиночное условие
mask = titanic['Age'] > 60
titanic[mask][['Name', 'Age', 'Sex']].head()

In [ ]:
# Несколько условий через & (И) и | (ИЛИ)
# Каждое условие ОБЯЗАТЕЛЬНО в скобках!

titanic[(titanic['Sex'] == 'female') & (titanic['Pclass'] == 1)][['Name', 'Age', 'Fare']].head()

In [ ]:
# isin — принадлежность списку
# Удобнее, чем цепочка из (x == a) | (x == b)
titanic[titanic['Pclass'].isin([1, 2])].shape

In [ ]:
# between — диапазон (включительно)
titanic[titanic['Age'].between(18, 25)][['Name', 'Age']].head(3)

In [ ]:
# query — альтернатива фильтрации, ближе к SQL
# Имена столбцов с пробелами оборачиваются в обратные кавычки

titanic.query("Age > 60 and Pclass == 1")[['Name', 'Age']].head()

In [ ]:
# str.contains — поиск подстроки в тексте
# na=False означает, что NaN не сломают фильтр
titanic[titanic['Name'].str.contains('Mrs', case=False, na=False)].head(3)

In [ ]:
# nlargest / nsmallest — топ-N без сортировки
# Быстрее, чем sort_values().head()
titanic.nlargest(3, 'Fare')[['Name', 'Fare']]

### ⚠️ Важное предупреждение: SettingWithCopyWarning

Когда вы фильтруете DataFrame и пытаетесь изменить результат, Pandas иногда не понимает — это копия или «вид» на оригинал. Всегда делайте `.copy()` перед изменениями.

In [ ]:
# ❌ Плохо — может вызвать предупреждение или баг
# adults = titanic[titanic['Age'] >= 18]
# adults['new_col'] = 1   # SettingWithCopyWarning!

# ✅ Правильно — явная копия
adults = titanic[titanic['Age'] >= 18].copy()
adults['status'] = 'взрослый'
adults.head(2)

## 7. Сортировка

In [ ]:
# По одному столбцу
titanic.sort_values('Fare', ascending=False)[['Name', 'Pclass', 'Fare']].head()

In [ ]:
# По нескольким столбцам с разными направлениями
titanic.sort_values(['Pclass', 'Age'], ascending=[True, False]).head()

## 8. Работа с пропусками (NaN)

Пропуски — обычная проблема в реальных данных. Главные инструменты: `isna`, `fillna`, `dropna`, `interpolate`.

In [ ]:
# Сколько пропусков в каждом столбце?
missing = titanic.isna().sum()
missing[missing > 0].sort_values(ascending=False)

In [ ]:
# Процент пропусков — помогает решить, удалять ли столбец
missing_pct = (titanic.isna().mean() * 100).round(2)
missing_pct[missing_pct > 0]

In [ ]:
# Заполнение пропусков
# Медиана устойчива к выбросам (в отличие от среднего)
# fillna НЕ меняет исходный DataFrame без присваивания!

titanic['Age'] = titanic['Age'].fillna(titanic['Age'].median())

# Для категориальных — моду (самое частое значение)
most_common_embarked = titanic['Embarked'].mode()[0]
titanic['Embarked'] = titanic['Embarked'].fillna(most_common_embarked)

# Cabin почти весь пустой — удалим столбец
if 'Cabin' in titanic.columns:
    titanic = titanic.drop(columns=['Cabin'])

print(f"Осталось пропусков: {titanic.isna().sum().sum()}")

In [ ]:
# interpolate — линейная интерполяция (полезна для временных рядов)
# Пример: заполнить пропуски средним соседей
s = pd.Series([1, np.nan, np.nan, 8])
print("Исходная:", s.tolist())
print("Интерполяция:", s.interpolate().tolist())

## 9. Дубликаты

In [ ]:
# Полные дубликаты строк
print("Полных дубликатов:", titanic.duplicated().sum())

# Дубликаты по подмножеству столбцов
print("Дубликатов по полу и классу:", titanic.duplicated(subset=['Sex', 'Pclass']).sum())

# Удаление (keep='first' оставляет первое вхождение)
# titanic = titanic.drop_duplicates(subset=['Name'], keep='first')

## 10. Типы данных и преобразования

Правильные типы ускоряют операции и экономят память. Особенно важны: `category` и `datetime`.

In [ ]:
# Текущие типы
titanic.dtypes

In [ ]:
# Преобразование в категорию — экономит память и ускоряет groupby
titanic['Sex'] = titanic['Sex'].astype('category')
titanic['Pclass'] = titanic['Pclass'].astype('category')
titanic['Embarked'] = titanic['Embarked'].astype('category')

print(titanic[['Sex', 'Pclass', 'Embarked']].dtypes)

In [ ]:
# Упорядоченная категория (например, для сортировки)
titanic['Pclass'] = pd.Categorical(titanic['Pclass'], categories=[3, 2, 1], ordered=True)
print(titanic['Pclass'].cat.categories)

In [ ]:
# rename — переименование столбцов
titanic = titanic.rename(columns={
    'SibSp': 'siblings_spouses',
    'Parch': 'parents_children'
})
titanic.head(2)

In [ ]:
# replace — замена значений
# titanic['Sex'] = titanic['Sex'].replace({'male': 'М', 'female': 'Ж'})

# astype — приведение типов
# titanic['Fare'] = titanic['Fare'].astype('float32')  # экономия памяти

## 11. Работа с датами (datetime)

Даты — отдельный и очень важный тип. Без него невозможна работа с временными рядами.

In [ ]:
# Создадим столбец с датой (синтетически, для демонстрации)
titanic['ticket_date'] = pd.to_datetime(
    '1912-04-' + (titanic.index % 10 + 1).astype(str)
)
titanic[['Name', 'ticket_date']].head()

In [ ]:
# Извлечение компонентов даты
titanic['ticket_month'] = titanic['ticket_date'].dt.month
titanic['ticket_day'] = titanic['ticket_date'].dt.day
titanic['ticket_weekday'] = titanic['ticket_date'].dt.day_name()

titanic[['ticket_date', 'ticket_month', 'ticket_day', 'ticket_weekday']].head()

In [ ]:
# Фильтрация по датам (dt accessor)
titanic[titanic['ticket_date'].dt.day > 5].head(2)

In [ ]:
# Разница между датами
reference = pd.Timestamp('1912-04-10')
titanic['days_before'] = (reference - titanic['ticket_date']).dt.days
titanic[['ticket_date', 'days_before']].head()

## 12. Новые столбцы и векторизация

**Золотое правило Pandas:** избегайте `apply` с `axis=1`, если можно сделать векторно — это в 10-100 раз быстрее.

In [ ]:
# ✅ Векторизованные операции — быстро
# titanic['Fare_rub'] = titanic['Fare'] * 90
# titanic['Age_double'] = titanic['Age'] * 2
# titanic['Fare_per_year'] = titanic['Fare'] / titanic['Age']

In [ ]:
# np.where — векторный if/else
# Создадим флаг: дорогой билет (>100) или нет
titanic['is_expensive'] = np.where(titanic['Fare'] > 100, 'дорогой', 'дешёвый')
titanic['is_expensive'].value_counts()

In [ ]:
# np.select — несколько условий (векторный elif)
conditions = [
    titanic['Fare'] < 20,
    titanic['Fare'] < 100,
    titanic['Fare'] >= 100
]
choices = ['низкий', 'средний', 'высокий']
titanic['fare_category'] = np.select(conditions, choices, default='неизвестно')
titanic['fare_category'].value_counts()

In [ ]:
# cut — равные интервалы (биннинг)
titanic['age_bin'] = pd.cut(titanic['Age'], bins=[0, 18, 35, 60, 100], labels=['ребёнок', 'молодой', 'взрослый', 'пожилой'])
titanic['age_bin'].value_counts()

In [ ]:
# qcut — квантильные интервалы (по количеству, а не по диапазону)
titanic['fare_quintile'] = pd.qcut(titanic['Fare'], q=5, labels=['Q1', 'Q2', 'Q3', 'Q4', 'Q5'])
titanic['fare_quintile'].value_counts()

In [ ]:
# apply — только когда векторно невозможно
# Например, сложная логика или работа со строками посимвольно

def extract_title(name):
    if 'Mr.' in name: return 'Mr'
    if 'Mrs.' in name: return 'Mrs'
    if 'Miss.' in name: return 'Miss'
    return 'Other'

# apply к Series (один столбец) — медленнее вектора, но допустимо
titanic['title'] = titanic['Name'].apply(extract_title)
titanic['title'].value_counts().head()

In [ ]:
# Строковые методы через .str (векторизованы!)
titanic['Name_upper'] = titanic['Name'].str.upper()
titanic['Name_short'] = titanic['Name'].str[:10]
titanic[['Name', 'Name_upper', 'Name_short']].head(2)

## 13. Группировка и агрегация

`groupby` разбивает данные на группы, агрегирует каждую и собирает результат.

In [ ]:
# Простая агрегация: средняя цена билета по классам
titanic.groupby('Pclass')['Fare'].mean().round(2)

In [ ]:
# agg — несколько функций сразу, с новыми именами
titanic.groupby('Sex').agg(
    count=('Name', 'count'),
    avg_age=('Age', 'mean'),
    max_fare=('Fare', 'max'),
    survived_rate=('Survived', 'mean')
).round(2)

In [ ]:
# Группировка по двум столбцам + несколько метрик
titanic.groupby(['Sex', 'Pclass']).agg(
    n=('Name', 'count'),
    avg_fare=('Fare', 'mean')
).round(2)

In [ ]:
# transform — агрегация «без потери строк»
# Например: добавить столбец со средним возрастом по полу для КАЖДОЙ строки

titanic['avg_age_by_sex'] = titanic.groupby('Sex')['Age'].transform('mean').round(1)
titanic[['Sex', 'Age', 'avg_age_by_sex']].head()

In [ ]:
# filter — отфильтровать группы по условию
# Оставить только те группы (по полу), где средняя цена билета > 50

titanic.groupby('Sex').filter(lambda x: x['Fare'].mean() > 50).head(3)

In [ ]:
# value_counts — самая частая агрегация
# normalize=True возвращает доли
titanic['Embarked'].value_counts(normalize=True).round(3)

## 14. Сводные таблицы и кросстабуляция

In [ ]:
# pivot_table — сводная таблица
# Средняя выживаемость: строки — пол, столбцы — класс
titanic.pivot_table(values='Survived', index='Sex', columns='Pclass', aggfunc='mean').round(2)

In [ ]:
# margins — итоги по строкам и столбцам
titanic.pivot_table(values='Survived', index='Sex', columns='Pclass', aggfunc='mean', margins=True).round(2)

In [ ]:
# crosstab — частотная таблица (под капотом pivot_table)
# Часто удобнее для подсчётов
pd.crosstab(titanic['Sex'], titanic['Pclass'], margins=True)

In [ ]:
# crosstab с нормализацией (доли)
pd.crosstab(titanic['Sex'], titanic['Pclass'], normalize='index').round(2)

## 15. Изменение формы таблиц: melt и pivot

In [ ]:
# melt — широкий формат → длинный (wide → long)
# Полезно для визуализации и последующего groupby

wide = titanic.head(3)[['Name', 'Age', 'Fare']]
print("Wide:")
display(wide)

long = wide.melt(id_vars=['Name'], var_name='metric', value_name='value')
print("
Long:")
display(long)

In [ ]:
# pivot — длинный формат → широкий (обратная операция)
long.pivot(index='Name', columns='metric', values='value').reset_index()

## 16. Объединение таблиц

Аналоги SQL JOIN: `merge` (по ключу) и `concat` (склейка).

In [ ]:
# merge — JOIN по ключу
ports = pd.DataFrame({
    'code': ['S', 'C', 'Q'],
    'port_name': ['Саутгемптон', 'Шербур', 'Квинстаун'],
    'country': ['UK', 'France', 'Ireland']
})
ports

In [ ]:
# how='left' — как SQL LEFT JOIN (сохраняем все строки левой таблицы)
titanic = titanic.merge(ports, left_on='Embarked', right_on='code', how='left')
titanic[['Name', 'Embarked', 'port_name', 'country']].head()

In [ ]:
# concat — склейка
# axis=0 (по умолчанию) — друг под другом, axis=1 — рядом (горизонтально)

top = titanic.head(2)
bottom = titanic.tail(2)
pd.concat([top, bottom], axis=0)

## 17. Визуализация средствами Pandas

Pandas строит графики через matplotlib. Для анализа — быстро и удобно.

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (8, 4)

# Гистограмма
titanic['Age'].plot.hist(bins=30, edgecolor='black', title='Распределение возраста')
plt.xlabel('Возраст')
plt.show()

In [ ]:
# Столбчатая диаграмма
titanic.groupby('Pclass')['Survived'].mean().plot.bar(
    title='Доля выживших по классам', ylabel='Доля', rot=0, color='steelblue')
plt.ylim(0, 1)
plt.show()

In [ ]:
# Boxplot — распределение цены по классам
titanic.boxplot(column='Fare', by='Pclass')
plt.suptitle('')  # убрать автоматический заголовок
plt.title('Распределение цены билета по классам')
plt.show()

In [ ]:
# Корреляционная матрица (только числовые столбцы)
numeric_cols = titanic.select_dtypes(include='number').columns
corr = titanic[numeric_cols].corr()
corr.style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1).format("{:.2f}")

## 18. Сохранение результатов

In [ ]:
# CSV — универсальный формат
titanic.to_csv('titanic_clean.csv', index=False)

# Excel — удобно для коллег
titanic.to_excel('titanic_clean.xlsx', index=False, sheet_name='Passengers')

# Без индекса — чтобы не создавать лишний unnamed столбец при повторном чтении
print("Сохранено!")

## 19. Продвинутые приёмы и best practices

In [ ]:
# 1. Метод-цепочка (method chaining) — читаемый код без промежуточных переменных
result = (
    titanic
    .assign(Fare_rub=lambda df: df['Fare'] * 90)
    .query("Age >= 18")
    .groupby('Sex')
    .agg(avg_fare=('Fare_rub', 'mean'), count=('Name', 'count'))
    .round(2)
)
result

In [ ]:
# 2. pipe — применение своей функции в цепочке

def log_shape(df, label):
    print(f"{label}: {df.shape}")
    return df

result = (
    titanic
    .pipe(log_shape, 'Начало')
    .query("Fare > 10")
    .pipe(log_shape, 'После фильтра')
)

In [ ]:
# 3. memory_usage — оценка потребления памяти
print(titanic.memory_usage(deep=True).sum() / 1024, "KB")

# Сравнение: object vs category
# object: ~KB, category: ~0.5 KB для этого столбца

In [ ]:
# 4. Итерация — последний курорт
# Никогда НЕ делайте: for i in range(len(df)): ...
# Если без цикла никак — используйте iterrows или itertuples (быстрее)

for idx, row in titanic.head(2).iterrows():
    print(f"{row['Name']}: {row['Age']} лет")

## 🎓 Шпаргалка

| Задача | Код |
|---|---|
| Прочитать CSV | `pd.read_csv('file.csv')` |
| Прочитать Excel | `pd.read_excel('file.xlsx')` |
| Первые строки | `df.head(n)` |
| Случайная выборка | `df.sample(n)` |
| Инфо о данных | `df.info()` |
| Статистика | `df.describe()` |
| Уникальные значения | `df.nunique()` |
| Выбор по меткам | `df.loc[rows, cols]` |
| Выбор по позициям | `df.iloc[rows, cols]` |
| Отфильтровать | `df[df['col'] > 5]` |
| Фильтр SQL-style | `df.query("col > 5")` |
| Отсортировать | `df.sort_values('col')` |
| Топ-N | `df.nlargest(5, 'col')` |
| Найти пропуски | `df.isna().sum()` |
| Заполнить пропуски | `df['col'].fillna(0)` |
| Удалить дубликаты | `df.drop_duplicates()` |
| Переименовать столбцы | `df.rename(columns={'old':'new'})` |
| Сменить тип | `df['col'].astype('category')` |
| Даты | `pd.to_datetime(df['col'])` |
| Новый столбец | `df['new'] = df['a'] * 2` |
| Векторный if/else | `np.where(cond, a, b)` |
| Биннинг | `pd.cut(df['col'], bins=...)` |
| Группировка | `df.groupby('col')['val'].mean()` |
| Агрегация | `df.groupby('col').agg(x=('val','mean'))` |
| Сводная таблица | `df.pivot_table(...)` |
| Частотная таблица | `pd.crosstab(a, b)` |
| Широкий → длинный | `df.melt(...)` |
| Соединить таблицы | `df1.merge(df2, on='key')` |
| Склеить таблицы | `pd.concat([df1, df2])` |
| Сохранить CSV | `df.to_csv('out.csv', index=False)` |

## 🏋️ Задачи для практики

1. Найдите самого молодого и самого старшего пассажира (`idxmin`, `idxmax`).
2. Посчитайте средний возраст выживших и погибших.
3. Определите, какой порт посадки (`Embarked`) давал самую высокую выживаемость.
4. Создайте столбец `FamilySize = siblings_spouses + parents_children + 1` и сравните выживаемость для одиночек (`FamilySize == 1`) и семей.
5. Постройте гистограмму цен билетов (`Fare`).
6. **Средний:** Используя `cut`, разбейте пассажиров на 4 возрастные группы и посчитайте выживаемость в каждой.
7. **Средний:** С помощью `pivot_table` постройте таблицу: строки — `Sex`, столбцы — `Pclass`, значения — средняя `Fare`.
8. **Продвинутый:** Найдите «аномалии»: пассажиров третьего класса, заплативших больше медианы первого класса.
9. **Продвинутый:** Создайте столбец `deck` из первой буквы `Cabin` (если `Cabin` есть в данных). Посчитайте выживаемость по палубам.

**Совет:** Открывайте ноутбук в Jupyter / VS Code / Google Colab, выполняйте ячейки по порядку и экспериментируйте!  
Удачи! 🚀